# League of Legends Pro Match Analysis - Data Overview

## 프로젝트 개요 

이 프로젝트는 Oracle's Elixir에서 제공하는 League of Legends 프로 경기 데이터를 활용하여 프로 경기에서 승리와 관련된 주요 요인을 통계적으로 분석하는 것이 목표이다.

### 주요 분석 관심사
- 블루/레드 진영에 따른 승률 차이
- First Blood, First Tower, First Dragon 등 초반 지표와 승리의 관계
- 오브젝트 획득과 승리의 관계
- 리그별 경기 시간 및 경기 양상의 차이
- 승리팀과 패배팀의 주요 지표 비교 분석

이 파일에서는 본격적인 분석 전에 데이터 구조를 확인하고 분석에 사용할 데이터를 가공한다.

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [90]:
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.5.0


**데이터 파일 확인**

In [91]:
DATA_DIR = Path("../data/raw")

csv_files = list(DATA_DIR.glob("*.csv"))
print(csv_files)

[WindowsPath('../data/raw/2025_LoL_esports_match_data_from_OraclesElixir.csv')]


**데이터 불러오기 및 크기 확인**

In [92]:
DATA_PATH = csv_files[0]
df = pd.read_csv(DATA_PATH, low_memory=False)
df.head()

,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,200.0,224.0,-1.0,17.0,1.0,1.0,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,157.0,-2363.0,-1444.0,-18.0,0.0,1.0,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,241.0,-1552.0,-2465.0,-41.0,1.0,0.0,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,257.0,-2613.0,-1156.0,-6.0,1.0,1.0,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,20.0,-662.0,-734.0,18.0,0.0,2.0,2.0,0.0,8.0,0.0


In [93]:
df.shape

(120456, 165)

In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120456 entries, 0 to 120455
Columns: 165 entries, gameid to opp_deathsat25
dtypes: float64(122), int64(20), str(23)
memory usage: 151.6 MB


In [95]:
df["position"].value_counts(dropna=False)

position
top     20076
jng     20076
mid     20076
bot     20076
sup     20076
team    20076
Name: count, dtype: int64

**팀 데이터만 분리**

In [96]:
team_df = df[df["position"]=="team"].copy()
team_df.shape

(20076, 165)

In [97]:
team_df.head()

,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
10,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,875.0,-6966.0,-5800.0,-30.0,3.0,5.0,10.0,10.0,26.0,3.0
11,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,845.0,6966.0,5800.0,30.0,10.0,26.0,3.0,3.0,5.0,10.0
22,LOLTMNT06_96134,complete,NaN,LFL2,2025,Winter,0,2025-01-11 12:06:37,1,15.01,...,735.0,8377.0,4306.0,99.0,15.0,35.0,8.0,7.0,12.0,15.0
23,LOLTMNT06_96134,complete,NaN,LFL2,2025,Winter,0,2025-01-11 12:06:37,1,15.01,...,834.0,-8377.0,-4306.0,-99.0,7.0,12.0,15.0,15.0,35.0,8.0
34,LOLTMNT06_95160,complete,NaN,LFL2,2025,Winter,0,2025-01-11 13:07:47,1,15.01,...,866.0,-5621.0,-4329.0,-107.0,14.0,25.0,14.0,14.0,32.0,14.0


**데이터 완성도 확인**

In [98]:
team_df["datacompleteness"].value_counts(dropna=False)

datacompleteness
complete    18442
partial      1634
Name: count, dtype: int64

In [99]:
team_df = team_df[team_df["datacompleteness"]=="complete"].copy()
team_df.shape

(18442, 165)

In [100]:
team_df.groupby("gameid").size().value_counts()
#게임별로 row가 두개인지 확인

2    9221
Name: count, dtype: int64

**리그별 경기 수 확인**

In [101]:
team_df["league"].value_counts().head(20)

league
LCK       1110
LCKC      1084
LAS        974
EM         878
LJL        748
LFL        636
LEC        612
PCS        604
PRM        602
CD         602
AL         586
LCP        582
NACL       550
LVP SL     528
NLC        498
TCL        480
RL         478
LFL2       466
LTA S      444
HLL        442
Name: count, dtype: int64

**2025년 데이터만 존재하는지 확인**

In [102]:
team_df["year"].value_counts().sort_index()

year
2025    17946
2026      496
Name: count, dtype: int64

**26년 데이터 제거**

In [103]:
team_df_2025= team_df[team_df["year"]==2025].copy()
team_df_2025.shape

(17946, 165)

In [104]:
analysis_df = team_df_2025.copy()

### 간단한 EDA

- 진영별 승률보기

In [106]:
analysis_df.groupby("side")["result"].mean()

side
Blue    0.532709
Red     0.467291
Name: result, dtype: float64

- 경기 시간 분포

In [ ]:
analysis_df["gamelength_min"] = analysis_df["gamelength"] / 60
# 경기 시간을 분 기준으로 한 geamelength_min column 생성
analysis_df["gamelength_min"].describe()

count    17946.000000
mean        32.000472
std          5.478996
min         17.650000
25%         28.233333
50%         30.916667
75%         35.050000
max         60.350000
Name: gamelength_min, dtype: float64

**4대 주요 리그만 추출**

In [109]:
major_leagues = ["LCK", "LPL", "LEC", "LCS"]

major_df = analysis_df[analysis_df["league"].isin(major_leagues)].copy()

major_df.shape

(1722, 166)